### Инициализация

In [ ]:
from llama_index.core import Settings
from qdrant_client import QdrantClient
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.llms.openai_like import OpenAILike
from pathlib import Path
import os
import time
import json
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
COLLECT_NAME = "wb"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

In [ ]:
results_dir = Path.cwd() / "results"

context_dir = Path.cwd() / "context"

In [9]:
# print(f"PyTorch: {torch.__version__}")
# print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# функция сохранения результата в файл
def save_result(result, file_name: str) -> None:
    result_path = results_dir.joinpath(file_name)
    result["time"] = str(datetime.datetime.now())
    with open(result_path, "w") as f:
        f.write(json.dumps(result) + "\n")

In [11]:
# специальные промпы для Qwen
def completion_to_prompt(completion):
   return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"

def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n" + prompt

    prompt = prompt + "<|im_start|>assistant\n"

    return prompt

In [12]:
# подключение embedding модели с CPU
Settings.embed_model = TextEmbeddingsInference(model_name=EMBED_MODEL_NAME,
                                               base_url="http://localhost:8081")

In [29]:
# подключение llm из vLLM
Settings.llm = OpenAILike(
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    model="qwen3-4b",                    
    api_base="http://localhost:8000/v1", 
    # api_key="dummy",                     
    temperature=0.7,
    max_tokens=1024,                    
    timeout=120,                        
    additional_kwargs={
        "top_p": 0.8,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0,
    }
)



In [30]:
import requests
try:
    req = requests.get(url="http://localhost:8000/health")
    req.status_code
    print("✅ vLLM успешно подключён к LlamaIndex")
except Exception as e:
    print(f"Ошибка {e}")

✅ vLLM успешно подключён к LlamaIndex


### BaseLine

In [ ]:
# подключаемся с Qdrant
client = QdrantClient(url="http://127.0.0.1:6333")

if client.collection_exists(collection_name=COLLECT_NAME):
    qdrant_store = QdrantVectorStore(collection_name=COLLECT_NAME,
                                     client=client)
    index = VectorStoreIndex.from_vector_store(qdrant_store)

In [20]:
print(index)

In [38]:
t0 = time.time()
query = "Кто такой Продавец?"
query_engine = index.as_query_engine()
response = query_engine.query(query)
print(response.response)

latency = time.time() - t0  # время выполнения

Продавец — Заказчик или иное лицо (коммерческая организация, индивидуальный предприниматель, или гражданин, признанный в установленном законом порядке самозанятым), которое осуществляет деятельность на Торговой площадке и с которым Клиент заключил договор розничной купли-продажи Товаров.


In [39]:
latency

1.5341284275054932

In [33]:
response.source_nodes

[NodeWithScore(node=TextNode(id_='2bd658a2-0a6f-4333-a970-5e50023c647a', embedding=None, metadata={'file_path': '/home/yuri/HSE/ВКР/Designing-a-Question-Answering-System-Based-on-the-RAG-Retrieval-Augmented-Generation-Architecture./data/оферта WB_1.txt', 'file_name': 'оферта WB_1.txt', 'file_type': 'text/plain', 'file_size': 1498439, 'creation_date': '2026-03-13', 'last_modified_date': '2026-03-13'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='3d0352f7-16fc-4c84-9742-457fb277f192', node_type='4', metadata={'file_path': '/home/yuri/HSE/ВКР/Designing-a-Question-Answering-System-Based-on-the-RAG-Retrieval-Augmented-Generation-Architecture./data/оферта WB_1.txt', 'file_name': 'оферта WB_1.txt', 'file_type':

### Тестовый датасет

Тестовый датасет создал с помощью сервиса NoteBookLM от Google следующим образом: загрузил документы оферты в сервис и задал промп: 
```
Особое задание - мне нужны эталонные вопросы - ответы по источникам. Напиши 30 вопросов - ответов, исключительно по источникам. Вопросы должны быть разными: от простых до сложных. Ответы должны быть исключительно по источникам, точные и верные, без лишней информации. Каждый вопрос - ответ на новой строке, пронумерованы. Больше ничего не пиши.
```

 Со слов Gemini в сервисе используется модель - Gemini 1.5 Pro

In [40]:
true_file_path = Path.cwd() / "data1"
file = true_file_path.joinpath("QuestionsWithTrueAnswers.txt")

with open(file, "r", encoding="utf-8") as f:
    questions_true_answers = f.readlines()

In [41]:
questions_true_answers

['\ufeff1. Что является моментом Активации ПВЗ? Моментом Активации ПВЗ признается получение Исполнителем подтверждения от Заказчика о возможной Активации. \n',
 '2. Какое минимальное количество камер видеонаблюдения требуется при стандартных условиях открытия ПВЗ? Должно быть установлено не менее 3 (трех) камер видеонаблюдения.\n',
 '3. Какова минимальная глубина хранения видеоархива для ПВЗ, расположенных вне Республики Беларусь? Минимальная глубина архива составляет 90 (девяносто) дней на каждой камере.\n',
 '4. Какой штраф предусмотрен за фиктивную выдачу товара, когда Исполнитель отметил выдачу на Портале без фактической передачи? Штраф составляет 100% от Стоимости Товара.\n',
 '5. Где должны находиться подготовленные Возвратные коробки с товарами для отправки на Склад? Подготовленные Возвратные коробки должны быть размещены в Клиентской зоне ПВЗ.\n',
 '6. В течение какого времени Исполнитель обязан вложить Товар в Возвратную коробку после отказа Клиента? Физическое вложение товара

In [42]:
questions = []
true_answers = []
for qa in questions_true_answers:
    qa = qa.strip()
    q_index = qa.find("?") + 1
    questions.append(qa[:q_index].strip().split(".")[1].strip())
    true_answers.append(qa[q_index:].strip())

In [43]:
questions

['Что является моментом Активации ПВЗ?',
 'Какое минимальное количество камер видеонаблюдения требуется при стандартных условиях открытия ПВЗ?',
 'Какова минимальная глубина хранения видеоархива для ПВЗ, расположенных вне Республики Беларусь?',
 'Какой штраф предусмотрен за фиктивную выдачу товара, когда Исполнитель отметил выдачу на Портале без фактической передачи?',
 'Где должны находиться подготовленные Возвратные коробки с товарами для отправки на Склад?',
 'В течение какого времени Исполнитель обязан вложить Товар в Возвратную коробку после отказа Клиента?',
 'При образовании очереди из какого количества человек Исполнитель считается нарушившим правила обслуживания?',
 'С какой периодичностью Исполнитель обязан проводить инвентаризацию товаров в ПВЗ?',
 'Какая максимальная продолжительность времени отведена на проведение одной инвентаризации?',
 'Разрешено ли использование камер видеонаблюдения с технологией передачи данных по Wi-Fi?',
 'Какой штраф налагается на Исполнителя за н

In [44]:
true_answers

['Моментом Активации ПВЗ признается получение Исполнителем подтверждения от Заказчика о возможной Активации.',
 'Должно быть установлено не менее 3 (трех) камер видеонаблюдения.',
 'Минимальная глубина архива составляет 90 (девяносто) дней на каждой камере.',
 'Штраф составляет 100% от Стоимости Товара.',
 'Подготовленные Возвратные коробки должны быть размещены в Клиентской зоне ПВЗ.',
 'Физическое вложение товара должно быть осуществлено в течение 24 (двадцати четырех) часов с момента отказа',
 'Исполнитель обязуется не допускать образования очереди от 6 (шести) и более Клиентов.',
 'Инвентаризация проводится не реже 1 (одного) раза в 30 (тридцать) календарных дней.',
 'Нормативная продолжительность времени проведения инвентаризации составляет 24 (двадцать четыре) часа с момента её начала.',
 'Нет, использование камер с Wi-Fi категорически запрещено, передача данных должна осуществляться только через Ethernet.',
 'Штраф составляет 10 000 (десять тысяч) российских рублей за каждое сле

In [45]:
import pandas as pd

results = []

for q in questions:
    response = query_engine.query(q) 
    
    results.append({
        "question": q,
        "answer": response.response,
        "contexts": [node.node.get_content() for node in response.source_nodes],
        "ground_truth": true_answers[questions.index(q)]
    })

df_eval = pd.DataFrame(results)

In [65]:
df_eval.to_csv(results_dir.joinpath("df_eval"))

In [ ]:
# df_eval = pd.read_csv("results/df_eval", index_col=0)

In [46]:
df_eval

,question,answer,contexts,ground_truth
0,Что является моментом Активации ПВЗ?,Моментом Активации ПВЗ является получение Испо...,[Размещение в одном строении/здании более одно...,Моментом Активации ПВЗ признается получение Ис...
1,Какое минимальное количество камер видеонаблюд...,При стандартных условиях открытия ПВЗ (все ост...,[Запрещено использовать камеры с Wi-Fi;\n— реж...,Должно быть установлено не менее 3 (трех) каме...
2,Какова минимальная глубина хранения видеоархив...,Минимальная глубина хранения видеоархива для П...,[Запрещено использовать камеры с Wi-Fi;\n— реж...,Минимальная глубина архива составляет 90 (девя...
3,Какой штраф предусмотрен за фиктивную выдачу т...,"Штраф за фиктивную выдачу товара, когда Исполн...",[15.\n5.20.\nИсполнитель обязуется выдать пост...,Штраф составляет 100% от Стоимости Товара.
4,Где должны находиться подготовленные Возвратны...,"Подготовленные Возвратные коробки с товарами, ...",[2.4.1. Подготовленные Возвратные коробки с То...,Подготовленные Возвратные коробки должны быть ...
5,В течение какого времени Исполнитель обязан вл...,"Информация о сроке, в течение которого Исполни...",[25.\n3.4.\nПри оказании услуг по подготовке Т...,Физическое вложение товара должно быть осущест...
6,При образовании очереди из какого количества ч...,Исполнитель считается нарушившим правила обслу...,"[), а также упаковки в сейф-пакет в тех случая...",Исполнитель обязуется не допускать образования...
7,С какой периодичностью Исполнитель обязан пров...,Исполнитель обязан проводить инвентаризацию то...,[Перемещение Товаров в альтернативный ПВЗ допу...,Инвентаризация проводится не реже 1 (одного) р...
8,Какая максимальная продолжительность времени о...,"Максимальная продолжительность времени, отведё...",[5.1.3. Заказчик вправе уведомить Исполнителя ...,Нормативная продолжительность времени проведен...
9,Разрешено ли использование камер видеонаблюден...,"Нет, использование камер видеонаблюдения с тех...",[9.1.4. Исполнитель обязан проверять ежедневно...,"Нет, использование камер с Wi-Fi категорически..."


In [56]:
from ragas.run_config import RunConfig


run_config = RunConfig(
    timeout=60, 
    # max_workers=2, # ограничиваем одним потоком (ограничения из-за локального запуска)
    max_retries=10
)

In [57]:
from ragas import evaluate
from datasets import Dataset
from ragas.llms import LlamaIndexLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper


ragas_llm = LlamaIndexLLMWrapper(llm=Settings.llm)
ragas_embeds = LlamaIndexEmbeddingsWrapper(embeddings=Settings.embed_model)

dataset = Dataset.from_pandas(df_eval)

score = evaluate(
    dataset,
    llm=ragas_llm, 
    embeddings=ragas_embeds,
    run_config=run_config 
)

print(score)

/tmp/ipykernel_342480/1410096092.py:7: DeprecationWarning: LlamaIndexLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LlamaIndexLLMWrapper(llm=Settings.llm)
/tmp/ipykernel_342480/1410096092.py:8: DeprecationWarning: LlamaIndexEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeds = LlamaIndexEmbeddingsWrapper(embeddings=Settings.embed_model)
Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]n values greater than 1 not support for LlamaIndex LLMs
n values greater than 1 not support for LlamaIndex LLMs
n values greater than 1 not support for LlamaIndex LLMs
n valu

{'answer_relevancy': 0.4879, 'context_precision': 0.6500, 'faithfulness': 0.8443, 'context_recall': 0.6667}


In [51]:
result = score.to_pandas()[["answer_relevancy", "context_precision", "faithfulness", "context_recall"]].agg("mean", axis=0)

In [52]:
result = result.to_dict()
result["latency"] = latency

In [53]:
save_result(result, "baseline_vllm_fp8")

In [ ]:
# from google.genai import Client
# import os
# from dotenv import load_dotenv

# load_dotenv()
# google_api_key = os.getenv("GEMINI_API_KEY")
# google_client = Client(api_key=google_api_key)

In [ ]:
# result = google_client.models.embed_content(
#         model="gemini-embedding-001",
#         contents= [
#             "What is the meaning of life?",
#             "What is the purpose of existence?",
#             "How do I bake a cake?"
#         ]
# )

# for embedding in result.embeddings:
#     print(len(embedding.values))